<a href="https://colab.research.google.com/github/SumitJainUTD/AI-projects/blob/main/lunarlanding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Q-Learning en el entorno LunarLander-v3

En primer lugar instalamos las dependencias necesarias para ejecutar el código. En este caso, necesitamos instalar la biblioteca `gymnasium` y su extensión `box2d` para poder utilizar el entorno LunarLander-v3:

In [9]:
!pip install gymnasium
!pip install gymnasium[box2d]


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


Realizamos los import necesarios, destacando en este caso la inclusión de `torch` para la implementación de la red neuronal que utilizaremos como función aproximadora en nuestro agente de Deep Q-Learning:

In [10]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from collections import deque

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)


Creamos la clase para la red neuronal que utilizaremos como función aproximadora en nuestro agente de Deep Q-Learning. Esta red tendrá una arquitectura simple con dos capas ocultas y una capa de salida que corresponde a las acciones posibles en el entorno:

In [11]:
class QNetwork(nn.Module):
    """
    Red neuronal para aproximar la función Q.

    Parámetros:
      - state_dim (int): Dimensión del estado.
      - action_dim (int): Número de acciones posibles.
      - hidden_dim (int): Número de neuronas en las capas ocultas.
    """

    def __init__(self, state_dim, action_dim, hidden_dim=64, seed = 42):
        super(QNetwork, self).__init__()
        # Primera capa: de estado a capa oculta de tamaño hidden_dim.
        self.seed = torch.manual_seed(seed)
        self.fc1 = nn.Linear(state_dim, hidden_dim)
        # Segunda capa oculta.
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        # Capa de salida: de hidden_dim a número de acciones.
        self.fc3 = nn.Linear(hidden_dim, action_dim)

    def forward(self, x):
        """
        Propagación hacia adelante.

        Parámetro:
          - x (Tensor): Estado de entrada con forma [batch_size, state_dim].

        Retorna:
          - Tensor: Valores Q para cada acción, con forma [batch_size, action_dim].
        """
        # Aplicar la primera capa seguida de ReLU.
        x = F.relu(self.fc1(x))
        # Aplicar la segunda capa seguida de ReLU.
        x = F.relu(self.fc2(x))
        # Capa de salida sin activación, para obtener los valores Q.
        x = self.fc3(x)
        return x
     


También necesitaremos una clase para gestionar la memoria de experiencia, que nos permitirá almacenar las transiciones y muestrear mini-batches para el entrenamiento de la red neuronal:

In [12]:

class DqnReplayBuffer:
    def __init__(self, max_capacity):
        # Configuramos el dispositivo de hardware disponible
        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        
        # Utilizamos deque para manejar el límite de capacidad automáticamente
        self.buffer = deque(maxlen=max_capacity)

    def push(self, transition):
        # Añadimos la nueva transición; si se supera max_capacity, deque borra la más antigua
        self.buffer.append(transition)

    def sample(self, batch_size):
        experiences = random.sample(self.buffer, k = batch_size)
        states = torch.from_numpy(np.vstack([e[0] for e in experiences if e is not None])).float().to(self.device)
        next_states = torch.from_numpy(np.vstack([e[1] for e in experiences if e is not None])).float().to(self.device)

        actions = torch.from_numpy(np.vstack([e[2] for e in experiences if e is not None])).long().to(self.device)
        rewards = torch.from_numpy(np.vstack([e[3] for e in experiences if e is not None])).float().to(self.device)
        dones = torch.from_numpy(np.vstack([e[4] for e in experiences if e is not None]).astype(np.uint8)).float().to(self.device)
        
        


        return states, next_states, actions, rewards, dones
    
    def __len__(self):
        return len(self.buffer)

In [13]:
class DQNAgent:
    def __init__(self, state_size, action_size, seed):
        self.state_size = state_size
        self.action_size = action_size
        self.seed = random.seed(seed)
        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

        # Hiperparametros de DQN
        self.buffer_size = int(1e5)  # Capacidad de la memoria
        self.batch_size = 100         # Tamaño del lote de entrenamiento
        self.gamma = 0.99            # Factor de descuento
        self.tau = 1e-3              # Parametro para la actualizacion suave de la red objetivo
        self.lr = 5e-4               # Tasa de aprendizaje
        self.update_every = 4        # Frecuencia de entrenamiento en pasos

        # Instanciamos la Red Local y la Red Objetivo
        self.qnetwork_local = QNetwork(state_size, action_size).to(self.device)
        self.qnetwork_target = QNetwork(state_size, action_size).to(self.device)
        

        # Definimos el optimizador para la red local
        self.optimizer = optim.Adam(self.qnetwork_local.parameters(), lr=self.lr)

        # Inicializamos la memoria de repeticion
        self.memory = DqnReplayBuffer(self.buffer_size)
        
        # Inicializamos el contador de pasos temporales
        self.t_step = 0

    def step(self, state, action, reward, next_state, done):
        # Guardamos la experiencia en la memoria
        self.memory.push((state, next_state, action, reward, done))
        
        # Incrementamos el contador de pasos
        self.t_step = (self.t_step + 1) % self.update_every
        
        # Aprendemos solo si ha pasado el intervalo y tenemos suficientes datos
        if self.t_step == 0 and len(self.memory) > self.batch_size:
            experiences = self.memory.sample(self.batch_size)
            self.learn(experiences)

    def get_action(self, state, eps=0.0):
        # Convertimos el estado de numpy a tensor
        state_tensor = torch.from_numpy(state).float().unsqueeze(0).to(self.device)
        
        # Ponemos la red en modo evaluacion temporalmente
        self.qnetwork_local.eval()
        with torch.no_grad():
            action_values = self.qnetwork_local(state_tensor)
        self.qnetwork_local.train()

        # Seleccion de accion epsilon-greedy
        if random.random() > eps:
            return np.argmax(action_values.cpu().data.numpy())
        else:
            return random.choice(np.arange(self.action_size))

    def learn(self, experiences):
        states, next_states, actions, rewards, dones = experiences

        # Obtenemos los valores Q maximos proyectados por la red objetivo para los estados siguientes
        Q_targets_next = self.qnetwork_target(next_states).detach().max(1)[0].unsqueeze(1)
        
        # Calculamos los objetivos Q reales (Target TD)
        Q_targets = rewards + (self.gamma * Q_targets_next * (1 - dones))

        # Obtenemos las predicciones de la red local para las acciones tomadas
        Q_expected = self.qnetwork_local(states).gather(1, actions)

        # Calculamos la perdida utilizando el Error Cuadratico Medio (MSE)
        loss = F.mse_loss(Q_expected, Q_targets)

        # Minimizamos la perdida actualizando los pesos de la red local
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        # Actualizamos la red objetivo mezclando suavemente sus pesos
        self.soft_update(self.qnetwork_local, self.qnetwork_target)

    def soft_update(self, local_model, target_model):
        # Interpolacion lineal: θ_target = τ*θ_local + (1 - τ)*θ_target
        for target_param, local_param in zip(target_model.parameters(), local_model.parameters()):
            target_param.data.copy_(self.tau * local_param.data + (1.0 - self.tau) * target_param.data)
            
    def test(self, env, n_episodes=5):
        total_wins = 0
        # Creamos una lista para almacenar las puntuaciones de todas las partidas
        all_scores = [] 
        
        for episode in range(1, n_episodes + 1):
            # Eliminamos la semilla para que cada partida de prueba sea un escenario nuevo
            state, _ = env.reset()
            done = False
            episode_reward = 0
            
            while not done:
                # Sin exploracion durante la prueba (politica puramente codiciosa)
                action = self.get_action(state, eps=0.0)  
                next_state, reward, terminated, truncated, _ = env.step(action)
                done = terminated or truncated
                
                episode_reward += reward
                state = next_state
                
            # Guardamos la puntuacion final del episodio completo
            all_scores.append(episode_reward)
            
            # Evaluamos la victoria sobre la recompensa ACUMULADA, fuera del bucle while
            if episode_reward >= 200.0: 
                total_wins += 1
                
            print(f"Episodio {episode}: Recompensa = {episode_reward:.2f}")
            
        # Calculamos la media real sumando todas las partidas
        media_real = sum(all_scores) / n_episodes
        print(f"\nResultados finales: {total_wins}/{n_episodes} victorias.")
        print(f"Recompensa media real: {media_real:.2f}")
            
            

In [14]:
def train_agent(env, agent, n_episodes=2000, max_t=1000, eps_start=1.0, eps_end=0.01, eps_decay=0.995):
    scores = []                        # Lista con todas las puntuaciones
    scores_window = deque(maxlen=100)  # Cola para calcular la media movil de los ultimos 100
    eps = eps_start                    # Inicializamos epsilon

    for i_episode in range(1, n_episodes + 1):
        state, _ = env.reset(seed=seed)
        score = 0
        
        # Iteramos los pasos del episodio
        for t in range(max_t):
            # Solicitamos la accion al agente
            action = agent.get_action(state, eps)
            next_state, reward, done, _, _ = env.step(action)
            
            # El agente procesa la transicion y aprende si es el momento
            agent.step(state, action, reward, next_state, done)
            
            state = next_state
            score += reward
            
            if done:
                break 
                
        # Guardamos las puntuaciones y reducimos epsilon
        scores_window.append(score)
        scores.append(score)
        eps = max(eps_end, eps_decay * eps)

        # Mostramos el progreso dinamicamente
        print(f'\rEpisodio {i_episode}\tMedia ultimos 100: {np.mean(scores_window):.2f}', end="")
        if i_episode % 100 == 0:
            print(f'\rEpisodio {i_episode}\tMedia ultimos 100: {np.mean(scores_window):.2f}')
            
        # Condicion de victoria formal del entorno
        if np.mean(scores_window) >= 200.0:
            print(f'\n¡Entorno resuelto en {i_episode - 100} episodios! Media: {np.mean(scores_window):.2f}')
            # Guardamos los pesos de la red local
            torch.save(agent.qnetwork_local.state_dict(), 'checkpoint_lunar_lander.pth')
            break
            
    return scores

In [15]:
import gymnasium as gym
env = gym.make('LunarLander-v3') 
state_shape = env.observation_space.shape
state_size = env.observation_space.shape[0]
number_actions = env.action_space.n
print('State shape: ', state_shape)
print('State size: ', state_size)
print('Number of actions: ', number_actions)

State shape:  (8,)
State size:  8
Number of actions:  4


In [16]:
number_episodes = 2000
maximum_number_timesteps_per_episode = 1000
epsilon_starting_value  = 1.0
epsilon_ending_value  = 0.01
epsilon_decay_value  = 0.985
epsilon = epsilon_starting_value
scores_on_100_episodes = deque(maxlen = 100)


agent = DQNAgent(state_size, number_actions, seed=seed)
scores = train_agent(env, 
                     agent, 
                     n_episodes=number_episodes, 
                     max_t=maximum_number_timesteps_per_episode, 
                     eps_start=epsilon_starting_value, 
                     eps_end=epsilon_ending_value,
                     eps_decay=epsilon_decay_value
                     )

Episodio 100	Media ultimos 100: -140.06
Episodio 200	Media ultimos 100: -95.632
Episodio 300	Media ultimos 100: -66.936
Episodio 400	Media ultimos 100: -47.40
Episodio 500	Media ultimos 100: -24.20
Episodio 600	Media ultimos 100: 6.2749
Episodio 700	Media ultimos 100: -11.68
Episodio 800	Media ultimos 100: 143.24
Episodio 824	Media ultimos 100: 200.29
¡Entorno resuelto en 724 episodios! Media: 200.29


In [20]:
# Instanciamos el agente (nace ignorante)
agent = DQNAgent(state_size, number_actions, seed=seed)


agent.qnetwork_local.load_state_dict(torch.load('checkpoint_lunar_lander.pth'))

# Lo ponemos en modo evaluacion estricto (buena practica en PyTorch)
agent.qnetwork_local.eval()
agent.test(env, n_episodes=100)

Episodio 1: Recompensa = -93.99
Episodio 2: Recompensa = 270.01
Episodio 3: Recompensa = 261.83
Episodio 4: Recompensa = -81.34
Episodio 5: Recompensa = 255.25
Episodio 6: Recompensa = 245.77
Episodio 7: Recompensa = 233.51
Episodio 8: Recompensa = -107.17
Episodio 9: Recompensa = 0.80
Episodio 10: Recompensa = -63.92
Episodio 11: Recompensa = -78.21
Episodio 12: Recompensa = 237.51
Episodio 13: Recompensa = 204.69
Episodio 14: Recompensa = -50.65
Episodio 15: Recompensa = 34.24
Episodio 16: Recompensa = -174.68
Episodio 17: Recompensa = -124.47
Episodio 18: Recompensa = 231.82
Episodio 19: Recompensa = 277.16
Episodio 20: Recompensa = 254.72
Episodio 21: Recompensa = 284.74
Episodio 22: Recompensa = -26.07
Episodio 23: Recompensa = 259.05
Episodio 24: Recompensa = -25.11
Episodio 25: Recompensa = -17.27
Episodio 26: Recompensa = 243.53
Episodio 27: Recompensa = 253.56
Episodio 28: Recompensa = 239.77
Episodio 29: Recompensa = -20.21
Episodio 30: Recompensa = -66.29
Episodio 31: Recomp

In [22]:
import glob
import io
import base64
import imageio
from IPython.display import HTML, display

def show_video_of_model(agent, env_name):
    env = gym.make(env_name, render_mode='rgb_array')
    state, _ = env.reset()
    done = False
    frames = []
    while not done:
        frame = env.render()
        frames.append(frame)
        action = agent.get_action(state, eps=0.0)  # Sin exploración durante la prueba
        state, reward, done, _, _ = env.step(action.item())
    env.close()
    imageio.mimsave('video.mp4', frames, fps=30)

show_video_of_model(agent, 'LunarLander-v3')

def show_video():
    mp4list = glob.glob('*.mp4')
    if len(mp4list) > 0:
        mp4 = mp4list[0]
        video = io.open(mp4, 'r+b').read()
        encoded = base64.b64encode(video)
        display(HTML(data='''<video alt="test" autoplay
                loop controls style="height: 400px;">
                <source src="data:video/mp4;base64,{0}" type="video/mp4" />
             </video>'''.format(encoded.decode('ascii'))))
    else:
        print("Could not find video")

show_video()

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (600, 400) to (608, 400) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).
